# MR Benchmark


## Setup

In [ ]:
import json
import os
import re
import math
import subprocess
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION — edit these for your benchmark run
# ============================================================
AGENT_OUT = Path("/home/yj348/gibbs/AI_agent/MR/<agent_name>")
PAPER_REF = Path("/home/yj348/gibbs/AI_agent/MR/paper_reference")

# Dataset under evaluation (1..7 per paper)
DATASET = "dataset1"  # change per evaluation pass

# The 16 methods that must be present
EXPECTED_METHODS = {
    "IVW_fe", "IVW", "dIVW", "Egger", "RAPS",
    "Weighted-median", "Weighted-mode", "MR-PRESSO", "MRMix",
    "cML-MA", "MR-Robust", "MR-Lasso", "MR-ConMix", "MR-CUE",
    "CAUSE", "MR-APSS"
}

# Four IV p-value thresholds expected
EXPECTED_THRESHOLDS = {5e-08, 5e-07, 5e-06, 5e-05}

# Required columns in every result file
RESULT_COLUMNS = ["exposure", "outcome", "method", "Threshold", "nsnp",
                  "beta", "se", "pval"]

# Required columns in every MRdat (clumped) file
MRDAT_COLUMNS = ["SNP", "A1", "A2", "b.exp", "se.exp", "b.out", "se.out",
                 "pval.exp", "Threshold"]

# Required columns in every formatted GWAS file
GWAS_COLUMNS = ["SNP", "CHR", "BP", "A1", "A2", "Z", "P", "N"]

# Tolerances
TOL_TYPE1_DELTA      = 0.02
TOL_BETA_DELTA       = 0.05
TOL_PVAL_LOG10_DELTA = 0.5

# Paths within the agent's output
DATASET_DIR = AGENT_OUT / DATASET
GWAS_DIR    = DATASET_DIR / "GWAS"
BG_DIR      = DATASET_DIR / "bg_paras"
MRDAT_DIR   = DATASET_DIR / "MRdat"
LOG_DIR     = DATASET_DIR / "logs"

print(f"Agent output:  {AGENT_OUT}")
print(f"Dataset:       {DATASET}")
print(f"Dataset path:  {DATASET_DIR}")


In [ ]:
# Results container and check helper
results = []

def check(check_id, name, condition, detail=""):
    """check_id is the verification-doc ID, e.g. '1A.01'."""
    status = "PASS" if condition else "FAIL"
    try:
        task = int(check_id.split('.')[0][0])
    except (TypeError, ValueError):
        task = 0
    results.append({"id": check_id, "task": task, "check": name,
                    "status": status, "detail": detail})
    icon = "✓" if condition else "✗"
    suffix = f" — {detail}" if detail else ""
    print(f"{icon} [{check_id}] {name}{suffix}")

def load_tsv_loose(path):
    """Load a result/data file with auto-detected separator, returning None on error."""
    try:
        return pd.read_csv(path, sep=None, engine='python')
    except Exception:
        try:
            return pd.read_csv(path, delim_whitespace=True)
        except Exception:
            return None

def load_tested_pairs():
    p = DATASET_DIR / "TestedPairs"
    if not p.exists():
        return None
    return load_tsv_loose(p)


## Task 1 — Method documentation and input validation

In [ ]:
print("=" * 70)
print("TASK 1 — Method documentation and input validation")
print("=" * 70)

# [1A.01] method_card.json exists, parses, lists all 16 methods
mc_path = AGENT_OUT / "method_card.json"
mc = None
if mc_path.exists():
    try:
        mc = json.loads(mc_path.read_text())
    except Exception as e:
        pass

methods_listed = set()
if isinstance(mc, dict):
    raw = mc.get("methods") or mc.get("Methods") or mc.get("method_list") or []
    if isinstance(raw, list):
        # Accept either list of strings or list of {name: ...}
        for item in raw:
            if isinstance(item, str):
                methods_listed.add(item)
            elif isinstance(item, dict) and 'name' in item:
                methods_listed.add(item['name'])

missing = EXPECTED_METHODS - methods_listed
check("1A.01", "method_card lists all 16 MR methods",
      mc is not None and len(missing) == 0,
      detail=f"missing: {sorted(missing)}" if missing else f"{len(methods_listed)} methods")

# [1A.02] Four IV thresholds + Cor.SelectionBias=True
if isinstance(mc, dict):
    th_raw = mc.get("iv_thresholds") or mc.get("IV_thresholds") or []
    declared = set(float(t) for t in th_raw) if isinstance(th_raw, list) else set()
    cor_sb = bool(mc.get("mr_apss_cor_selection_bias") or
                  mc.get("Cor.SelectionBias") or
                  mc.get("cor_selection_bias"))
else:
    declared = set()
    cor_sb = False
check("1A.02", "method_card declares 4 IV thresholds AND Cor.SelectionBias=TRUE",
      declared == EXPECTED_THRESHOLDS and cor_sb,
      detail=f"thresholds={sorted(declared)}, cor_sb={cor_sb}")

# [1A.03] Per-method package/version/output spec
ok = isinstance(mc, dict) and "method_details" in mc and \
     isinstance(mc["method_details"], dict)
if ok:
    sample = next(iter(mc["method_details"].values()), {})
    ok = all(k in sample for k in ["package", "version", "estimate_col",
                                    "se_col", "pval_col"])
check("1A.03", "Per-method package/version/output spec recorded",
      ok, detail="check method_details block in method_card")

# [1A.04] TestedPairs valid
pairs = load_tested_pairs()
ok = (pairs is not None and
      set(["exposure", "outcome"]).issubset(pairs.columns) and
      pairs.shape[0] > 0 and
      pairs[["exposure", "outcome"]].duplicated().sum() == 0 and
      (pairs["exposure"] != pairs["outcome"]).all())
check("1A.04", "TestedPairs has valid structure",
      ok, detail=f"{pairs.shape[0] if pairs is not None else 0} pairs")

# [1A.05] Every GWAS file referenced by TestedPairs exists
missing_gwas = []
if pairs is not None:
    traits = set(pairs["exposure"]).union(set(pairs["outcome"]))
    for t in traits:
        if not (GWAS_DIR / t).exists():
            missing_gwas.append(t)
check("1A.05", "Every GWAS file referenced by TestedPairs exists",
      len(missing_gwas) == 0, detail=f"{len(missing_gwas)} missing")

# [1A.06] LDSC LD-score reference for 22 autosomes
ldsc_dir = AGENT_OUT / "eur_w_ld_chr"
if not ldsc_dir.exists():
    ldsc_dir = DATASET_DIR / "eur_w_ld_chr"
present_chrs = set()
if ldsc_dir.exists():
    for f in ldsc_dir.iterdir():
        m = re.search(r"(\d+)\.l2", f.name)
        if m:
            present_chrs.add(int(m.group(1)))
check("1A.06", "LDSC LD-score reference covers chr 1..22",
      present_chrs >= set(range(1, 23)),
      detail=f"missing chr: {sorted(set(range(1,23)) - present_chrs)}")

# [1A.07] HapMap 3 SNP list
hm3_candidates = [AGENT_OUT / "snps.hm3.RData",
                  AGENT_OUT / "snps.hm3.tsv",
                  DATASET_DIR / "snps.hm3.RData"]
has_hm3 = any(p.exists() for p in hm3_candidates)
check("1A.07", "HapMap 3 SNP list exists", has_hm3)

# [1A.08] UK10K LDhm3.RDS for MR-CUE (only if MR-CUE in plan)
if "MR-CUE" in methods_listed:
    uk10k_dir_candidates = [AGENT_OUT / "UK10K", AGENT_OUT / "dataset_mrcue",
                            DATASET_DIR / "UK10K"]
    found_dir = next((d for d in uk10k_dir_candidates if d.exists()), None)
    present = set()
    if found_dir:
        for f in found_dir.iterdir():
            m = re.search(r"CHR(\d+)", f.name)
            if m:
                present.add(int(m.group(1)))
    check("1A.08", "UK10K LDhm3.RDS per-chromosome panels exist (for MR-CUE)",
          present >= set(range(1, 23)),
          detail=f"missing chr: {sorted(set(range(1,23)) - present)}")

# [1A.09] Delimiter detection — sample one GWAS file
if pairs is not None and pairs.shape[0] > 0:
    sample_trait = pairs.iloc[0]["exposure"]
    sample_path = GWAS_DIR / sample_trait
    if sample_path.exists():
        # Check whether the agent's logs record a detected delimiter
        # OR whether the file is loadable with auto-detection
        df = load_tsv_loose(sample_path)
        ok = df is not None and df.shape[1] >= len(GWAS_COLUMNS)
        check("1A.09", "GWAS files load with auto-detected delimiter",
              ok, detail=f"{sample_trait}: {df.shape if df is not None else 'unreadable'}")

# [1A.10] GWAS header validation
header_ok = True
header_problems = []
if pairs is not None:
    for trait in set(pairs["exposure"]).union(set(pairs["outcome"])):
        p = GWAS_DIR / trait
        if p.exists():
            df = load_tsv_loose(p)
            if df is not None:
                missing_cols = [c for c in GWAS_COLUMNS if c not in df.columns]
                if missing_cols:
                    header_ok = False
                    header_problems.append((trait, missing_cols))
check("1A.10", "All formatted GWAS files have required columns",
      header_ok and len(header_problems) == 0,
      detail=f"{len(header_problems)} files with missing columns")


## Task 2 — GWAS quality control and formatting

In [ ]:
print()
print("=" * 70)
print("TASK 2 — GWAS quality control and formatting")
print("=" * 70)

# Load HapMap 3 SNP list (best effort — accept .tsv or text file)
hm3_snps = set()
for cand in [AGENT_OUT / "snps.hm3.tsv", DATASET_DIR / "snps.hm3.tsv"]:
    if cand.exists():
        try:
            df = pd.read_csv(cand, sep=None, engine='python')
            col = "SNP" if "SNP" in df.columns else df.columns[0]
            hm3_snps = set(df[col].astype(str))
            break
        except Exception:
            pass
if not hm3_snps:
    print("  (HapMap 3 SNP list not available as TSV — check [2A.02] is informational)")

pairs = load_tested_pairs()
if pairs is None:
    check("2A.01", "Cannot validate Task 2 without TestedPairs",
          False, "TestedPairs not loaded")
else:
    traits = sorted(set(pairs["exposure"]).union(set(pairs["outcome"])))

    # [2A.01] Column order
    bad_order = []
    for t in traits:
        p = GWAS_DIR / t
        if p.exists():
            df = load_tsv_loose(p)
            if df is not None and list(df.columns)[:len(GWAS_COLUMNS)] != GWAS_COLUMNS:
                bad_order.append(t)
    check("2A.01", "Formatted GWAS files have exact column order",
          len(bad_order) == 0, detail=f"{len(bad_order)} mis-ordered")

    # [2A.02] HapMap 3 restriction
    if hm3_snps:
        bad_hm3 = []
        for t in traits:
            p = GWAS_DIR / t
            if p.exists():
                df = load_tsv_loose(p)
                if df is not None and "SNP" in df.columns:
                    out_of_hm3 = (~df["SNP"].astype(str).isin(hm3_snps)).sum()
                    if out_of_hm3 > 0:
                        bad_hm3.append((t, out_of_hm3))
        check("2A.02", "Every SNP is in HapMap 3 list",
              len(bad_hm3) == 0, detail=f"{len(bad_hm3)} traits violate")

    # [2A.03] Allele in {A,C,G,T}
    bad_alleles = []
    for t in traits:
        p = GWAS_DIR / t
        if not p.exists():
            continue
        df = load_tsv_loose(p)
        if df is None: continue
        for col in ("A1", "A2"):
            if col in df.columns:
                ok = df[col].astype(str).isin({"A","C","G","T"})
                if not ok.all():
                    bad_alleles.append((t, col, (~ok).sum()))
    check("2A.03", "All A1/A2 alleles are in {A,C,G,T}",
          len(bad_alleles) == 0, detail=f"{len(bad_alleles)} violations")

    # [2A.04] No ambiguous or identical-allele entries
    ambiguous_pairs = {("A","T"),("T","A"),("C","G"),("G","C"),
                       ("A","A"),("T","T"),("C","C"),("G","G")}
    bad_amb = []
    for t in traits:
        p = GWAS_DIR / t
        if not p.exists(): continue
        df = load_tsv_loose(p)
        if df is None or "A1" not in df.columns or "A2" not in df.columns:
            continue
        pairs_seen = set(zip(df["A1"].astype(str), df["A2"].astype(str)))
        bad = pairs_seen & ambiguous_pairs
        if bad:
            bad_amb.append((t, bad))
    check("2A.04", "No ambiguous or identical-allele SNPs remain",
          len(bad_amb) == 0, detail=f"{len(bad_amb)} traits have ambiguous SNPs")

    # [2A.05] MAF >= 0.05 if freq available
    bad_maf = []
    for t in traits:
        p = GWAS_DIR / t
        if not p.exists(): continue
        df = load_tsv_loose(p)
        if df is None: continue
        if "freq" in df.columns:
            maf = df["freq"].apply(lambda x: min(x, 1 - x) if pd.notna(x) else np.nan)
            n_bad = (maf < 0.05).sum()
            if n_bad > 0:
                bad_maf.append((t, n_bad))
    check("2A.05", "If freq present, no SNP has MAF < 0.05",
          len(bad_maf) == 0, detail=f"{len(bad_maf)} violations")

    # [2A.06] INFO >= 0.9 if info available
    bad_info = []
    for t in traits:
        p = GWAS_DIR / t
        if not p.exists(): continue
        df = load_tsv_loose(p)
        if df is None: continue
        if "info" in df.columns:
            n_bad = (df["info"] < 0.9).sum()
            if n_bad > 0:
                bad_info.append((t, n_bad))
    check("2A.06", "If info present, no SNP has INFO < 0.9",
          len(bad_info) == 0, detail=f"{len(bad_info)} violations")

    # [2A.07] MHC region excluded
    bad_mhc = []
    for t in traits:
        p = GWAS_DIR / t
        if not p.exists(): continue
        df = load_tsv_loose(p)
        if df is None or "CHR" not in df.columns or "BP" not in df.columns:
            continue
        in_mhc = ((df["CHR"].astype(str) == "6") &
                  (df["BP"] >= 26_000_000) & (df["BP"] <= 34_000_000))
        if in_mhc.sum() > 0:
            bad_mhc.append((t, int(in_mhc.sum())))
    check("2A.07", "MHC region (chr6 26-34 Mb) is excluded",
          len(bad_mhc) == 0, detail=f"{len(bad_mhc)} traits have MHC SNPs")

    # [2A.08] chi^2 <= max(N/1000, 80)
    bad_chi2 = []
    for t in traits:
        p = GWAS_DIR / t
        if not p.exists(): continue
        df = load_tsv_loose(p)
        if df is None or "Z" not in df.columns or "N" not in df.columns:
            continue
        chi2 = df["Z"] ** 2
        threshold = df["N"].apply(lambda n: max(n / 1000.0, 80.0))
        n_bad = (chi2 > threshold).sum()
        if n_bad > 0:
            bad_chi2.append((t, int(n_bad)))
    check("2A.08", "No SNP has chi^2 > max(N/1000, 80)",
          len(bad_chi2) == 0, detail=f"{len(bad_chi2)} traits violate")

# [2A.09] qc_log.tsv consistency
qc_log = DATASET_DIR / "qc_log.tsv"
if qc_log.exists():
    df = load_tsv_loose(qc_log)
    if df is not None and {"filter_name","n_in","n_dropped","n_out"}.issubset(df.columns):
        bad_rows = ((df["n_in"] - df["n_dropped"]) != df["n_out"]).sum()
        check("2A.09", "qc_log.tsv rows: n_in - n_dropped == n_out",
              bad_rows == 0, detail=f"{bad_rows} inconsistent rows")
    else:
        check("2A.09", "qc_log.tsv has required columns", False)
else:
    check("2A.09", "qc_log.tsv exists", False, str(qc_log))

# [2A.10] qc_summary.json with per-trait counts
qc_summary = DATASET_DIR / "qc_summary.json"
if qc_summary.exists():
    try:
        qs = json.loads(qc_summary.read_text())
        # Compare reported counts to actual row counts
        mismatches = []
        per_trait = qs.get("per_trait", qs)
        if isinstance(per_trait, dict):
            for trait, info in per_trait.items():
                reported = info.get("n_final") if isinstance(info, dict) else None
                p = GWAS_DIR / trait
                if reported is not None and p.exists():
                    df = load_tsv_loose(p)
                    actual = len(df) if df is not None else None
                    if actual is not None and abs(reported - actual) > 1:
                        mismatches.append((trait, reported, actual))
        check("2A.10", "qc_summary.json counts match actual file row counts",
              len(mismatches) == 0,
              detail=f"{len(mismatches)} mismatches")
    except Exception as e:
        check("2A.10", "qc_summary.json parses", False, str(e))
else:
    check("2A.10", "qc_summary.json exists", False)


## Task 3 — Background parameter estimation for MR-APSS

In [ ]:
print()
print("=" * 70)
print("TASK 3 — Background parameter estimation")
print("=" * 70)

pairs = load_tested_pairs()

# [3A.01] Omega and C files exist per pair
missing_bg = []
if pairs is not None and BG_DIR.exists():
    for _, row in pairs.iterrows():
        exp_, out_ = row["exposure"], row["outcome"]
        omega_path = BG_DIR / f"{exp_}~{out_}_Omega"
        c_path     = BG_DIR / f"{exp_}~{out_}_C"
        # Allow optional prefix like "1kgRef_"
        candidates_omega = [omega_path,
                            BG_DIR / f"1kgRef_{exp_}~{out_}_Omega"]
        candidates_c     = [c_path,
                            BG_DIR / f"1kgRef_{exp_}~{out_}_C"]
        has_omega = any(p.exists() for p in candidates_omega)
        has_c     = any(p.exists() for p in candidates_c)
        if not (has_omega and has_c):
            missing_bg.append((exp_, out_, has_omega, has_c))
check("3A.01", "Omega and C files exist for every pair",
      len(missing_bg) == 0, detail=f"{len(missing_bg)} missing")

# [3A.02] Each file parses as one-row tab-separated 4-value
def load_2x2(path_candidates):
    for p in path_candidates:
        if p.exists():
            try:
                df = pd.read_csv(p, sep=r"\s+", header=None)
                # Some files concatenate multiple rows; take the last
                vec = df.iloc[-1].values.astype(float)
                if len(vec) == 4:
                    return vec.reshape(2, 2)
                if len(vec) >= 4:
                    return vec[:4].reshape(2, 2)
            except Exception:
                continue
    return None

parsed_ok = 0
parse_failures = 0
omegas = {}
cs = {}
if pairs is not None and BG_DIR.exists():
    for _, row in pairs.iterrows():
        exp_, out_ = row["exposure"], row["outcome"]
        omega_files = [BG_DIR / f"{exp_}~{out_}_Omega",
                       BG_DIR / f"1kgRef_{exp_}~{out_}_Omega"]
        c_files     = [BG_DIR / f"{exp_}~{out_}_C",
                       BG_DIR / f"1kgRef_{exp_}~{out_}_C"]
        om = load_2x2(omega_files)
        cc = load_2x2(c_files)
        if om is not None and cc is not None:
            parsed_ok += 1
            omegas[(exp_, out_)] = om
            cs[(exp_, out_)] = cc
        else:
            parse_failures += 1
check("3A.02", "Omega and C files parse as 2x2 matrices",
      parse_failures == 0,
      detail=f"{parsed_ok} OK, {parse_failures} unparseable")

# [3A.03] Omega symmetric
asymmetric = [k for k, m in omegas.items() if abs(m[0,1] - m[1,0]) >= 1e-6]
check("3A.03", "Omega is symmetric within 1e-6 for every pair",
      len(asymmetric) == 0,
      detail=f"{len(asymmetric)} asymmetric")

# [3A.04] Omega positive semi-definite
not_psd = []
for k, m in omegas.items():
    eigs = np.linalg.eigvalsh((m + m.T) / 2)
    if eigs.min() < -1e-6:
        not_psd.append((k, float(eigs.min())))
check("3A.04", "Omega is positive semi-definite",
      len(not_psd) == 0,
      detail=f"{len(not_psd)} non-PSD")

# [3A.05] C diagonal entries in [0.5, 5]
bad_c = []
for k, m in cs.items():
    if not (0.5 <= m[0,0] <= 5 and 0.5 <= m[1,1] <= 5):
        bad_c.append((k, float(m[0,0]), float(m[1,1])))
check("3A.05", "C diagonal (LDSC intercepts) in [0.5, 5]",
      len(bad_c) == 0,
      detail=f"{len(bad_c)} out of range")

# [3A.06] LDSC was used with h2.fix.intercept=FALSE — check the recorded call
bg_script_candidates = [AGENT_OUT / "scripts" / "step2_background_paras.R",
                        AGENT_OUT / "scripts" / "step2_background_paras.py",
                        AGENT_OUT / "scripts" / "IV_selection.R"]
script_text = ""
for p in bg_script_candidates:
    if p.exists():
        script_text += p.read_text() + "\n"
check("3A.06", "LDSC step used h2.fix.intercept=FALSE",
      ("h2.fix.intercept = F" in script_text or
       "h2.fix.intercept=FALSE" in script_text or
       "h2_fix_intercept=False" in script_text),
      detail="searched step2 / IV_selection scripts")


## Task 4 — IV selection via PLINK LD clumping

In [ ]:
print()
print("=" * 70)
print("TASK 4 — IV selection via PLINK LD clumping")
print("=" * 70)

pairs = load_tested_pairs()

# [4A.01] MRdat file per pair
mrdats = {}
missing_mrdat = []
skipped = []
if pairs is not None and MRDAT_DIR.exists():
    for _, row in pairs.iterrows():
        exp_, out_ = row["exposure"], row["outcome"]
        candidates = [MRDAT_DIR / f"{exp_}~{out_}",
                      MRDAT_DIR / f"1kgRef_{exp_}~{out_}"]
        found = next((p for p in candidates if p.exists()), None)
        if found is None:
            missing_mrdat.append((exp_, out_))
        else:
            df = load_tsv_loose(found)
            mrdats[(exp_, out_)] = df

# Distinguish "missing without log" from "skipped with log"
skip_log = LOG_DIR / "skipped_pairs.log"
if skip_log.exists():
    logged_skips = set()
    for line in skip_log.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 2:
            logged_skips.add((parts[0], parts[1]))
    missing_mrdat = [p for p in missing_mrdat if p not in logged_skips]

check("4A.01", "MRdat file present (or pair explicitly logged as skipped)",
      len(missing_mrdat) == 0,
      detail=f"{len(missing_mrdat)} unaccounted-for")

# [4A.02] Required columns
bad_cols = []
for k, df in mrdats.items():
    if df is None:
        bad_cols.append(k)
        continue
    missing = [c for c in MRDAT_COLUMNS if c not in df.columns]
    if missing:
        bad_cols.append((k, missing))
check("4A.02", "MRdat files have required columns",
      len(bad_cols) == 0, detail=f"{len(bad_cols)} files have issues")

# [4A.03] pval.exp <= 5e-05 in every row
bad_pval = []
for k, df in mrdats.items():
    if df is None or "pval.exp" not in df.columns:
        continue
    n_bad = (df["pval.exp"] > 5e-05).sum()
    if n_bad > 0:
        bad_pval.append((k, int(n_bad)))
check("4A.03", "Every MRdat row has pval.exp <= 5e-05",
      len(bad_pval) == 0, detail=f"{len(bad_pval)} pairs violate")

# [4A.04] Threshold column equals 5e-05
bad_thresh = []
for k, df in mrdats.items():
    if df is None or "Threshold" not in df.columns:
        continue
    if not (df["Threshold"] == 5e-05).all():
        bad_thresh.append(k)
check("4A.04", "MRdat Threshold column equals 5e-05",
      len(bad_thresh) == 0, detail=f"{len(bad_thresh)} pairs violate")

# [4A.05] PLINK clumping parameters: 1000kb and 0.001
script_candidates = [AGENT_OUT / "scripts" / "step3_clump_IV.R",
                     AGENT_OUT / "scripts" / "IV_selection.R"]
text = ""
for p in script_candidates:
    if p.exists():
        text += p.read_text() + "\n"
has_kb = "clump_kb = 1000" in text or "clump_kb=1000" in text
has_r2 = ("clump_r2" not in text or "0.001" in text)  # r^2 default in clump() is 0.001
check("4A.05", "LD clumping uses kb=1000 and r^2=0.001",
      has_kb and has_r2, detail=f"kb={has_kb}, r2_consistent={has_r2}")

# [4A.06] No duplicate SNPs
dup_pairs = []
for k, df in mrdats.items():
    if df is None or "SNP" not in df.columns:
        continue
    if df["SNP"].duplicated().any():
        dup_pairs.append((k, int(df["SNP"].duplicated().sum())))
check("4A.06", "No duplicate SNPs within any MRdat file",
      len(dup_pairs) == 0, detail=f"{len(dup_pairs)} pairs have dups")

# [4A.07] b.exp and b.out finite for >= 3 IV pairs
bad_finite = []
for k, df in mrdats.items():
    if df is None: continue
    if len(df) < 3: continue
    if "b.exp" in df.columns and "b.out" in df.columns:
        if not (np.isfinite(df["b.exp"]).all() and np.isfinite(df["b.out"]).all()):
            bad_finite.append(k)
check("4A.07", "b.exp and b.out are finite for every MRdat row",
      len(bad_finite) == 0, detail=f"{len(bad_finite)} pairs have non-finite")

# [4A.08] Pairs with < 3 IVs logged with reason
n_too_few = sum(1 for k, df in mrdats.items() if df is not None and len(df) < 3)
n_logged_too_few = 0
if skip_log.exists():
    for line in skip_log.read_text().splitlines():
        if "fewer than 3 IVs" in line.lower() or "n_iv < 3" in line.lower():
            n_logged_too_few += 1
check("4A.08", "Pairs with < 3 IVs are explicitly logged",
      n_too_few <= n_logged_too_few,
      detail=f"{n_too_few} too-few cases, {n_logged_too_few} logged")


## Task 5 — Run the 16 MR methods

In [ ]:
print()
print("=" * 70)
print("TASK 5 — Run the 16 MR methods")
print("=" * 70)

# Result files expected per the task spec
expected_files = {
    "MRAPSS.MRres":    {"MR-APSS"},
    "MRmethods.MRres": {"IVW_fe", "IVW", "RAPS", "Egger", "MRMix",
                        "Weighted-median", "Weighted-mode", "MR-Robust",
                        "MR-Lasso", "cML-MA", "MR-PRESSO"},
    "ConMix.MRres":    {"MR-ConMix"},
    "dIVW.MRres":      {"dIVW"},
    "CAUSE.MRres":     {"CAUSE"},
    "MRCUE.MRres":     {"MR-CUE"},
}

# [5A.01..5A.05] result files exist
file_dfs = {}
for fname, methods_in_file in expected_files.items():
    path = DATASET_DIR / fname
    df = None
    if path.exists():
        # files are written with append + col.names=F, so usually no header
        # Try with header first, fall back to no header
        try:
            df = pd.read_csv(path, sep=r"\s+", header=None)
            df.columns = RESULT_COLUMNS[:df.shape[1]]
        except Exception:
            df = None
    file_dfs[fname] = df

check("5A.01", "MRAPSS.MRres exists with rows",
      file_dfs.get("MRAPSS.MRres") is not None and len(file_dfs["MRAPSS.MRres"]) > 0)
check("5A.02", "MRmethods.MRres exists with rows",
      file_dfs.get("MRmethods.MRres") is not None and len(file_dfs["MRmethods.MRres"]) > 0)
check("5A.03a", "ConMix.MRres exists with rows",
      file_dfs.get("ConMix.MRres") is not None and len(file_dfs["ConMix.MRres"]) > 0)
check("5A.03b", "dIVW.MRres exists with rows",
      file_dfs.get("dIVW.MRres") is not None and len(file_dfs["dIVW.MRres"]) > 0)
check("5A.04", "CAUSE.MRres exists with rows",
      file_dfs.get("CAUSE.MRres") is not None and len(file_dfs["CAUSE.MRres"]) > 0)
# MR-CUE may be skipped if UK10K panels absent
mrcue_required = ("MR-CUE" in EXPECTED_METHODS)
mrcue_skip_log = (LOG_DIR / "mrcue_skipped.log").exists()
mrcue_ok = (file_dfs.get("MRCUE.MRres") is not None and len(file_dfs["MRCUE.MRres"]) > 0) or mrcue_skip_log
check("5A.05", "MRCUE.MRres exists OR skip is logged",
      mrcue_ok)

# Concatenate all rows
all_rows = pd.concat([df for df in file_dfs.values() if df is not None],
                    ignore_index=True) if any(df is not None for df in file_dfs.values()) else None

if all_rows is None or len(all_rows) == 0:
    check("5A.06", "Result files have correct columns", False, "no rows to check")
else:
    # [5A.06] columns and order
    has_columns = all(c in all_rows.columns for c in RESULT_COLUMNS)
    check("5A.06", "Result rows have required columns",
          has_columns, detail=f"columns: {list(all_rows.columns)[:9]}")

    if has_columns:
        # [5A.07] union of methods == EXPECTED_METHODS
        observed = set(all_rows["method"].astype(str).unique())
        # Normalize naming variants
        norm = {m.replace("Inverse variance weighted", "IVW")
                 .replace("MR-IVW", "IVW")
                 .replace("MR-Egger", "Egger") for m in observed}
        missing = EXPECTED_METHODS - norm
        check("5A.07", "All 16 methods produce rows across result files",
              len(missing) == 0,
              detail=f"missing: {sorted(missing)}")

        # [5A.08] Thresholds restricted to expected set
        observed_th = set(float(t) for t in all_rows["Threshold"].dropna().unique())
        unexpected = observed_th - EXPECTED_THRESHOLDS
        check("5A.08", "Threshold column only contains {5e-08,5e-07,5e-06,5e-05}",
              len(unexpected) == 0,
              detail=f"unexpected: {sorted(unexpected)}")

        # [5A.09] Every (method) has all 4 thresholds represented
        method_thresh = defaultdict(set)
        for _, row in all_rows.iterrows():
            m = str(row["method"])
            try:
                method_thresh[m].add(float(row["Threshold"]))
            except (TypeError, ValueError):
                pass
        # MR-APSS may report one threshold or four depending on output convention
        incomplete = []
        for m, ts in method_thresh.items():
            if "APSS" in m.upper():
                continue
            if ts != EXPECTED_THRESHOLDS and len(ts) < 4:
                incomplete.append((m, sorted(ts)))
        check("5A.09", "Every method covers all four IV thresholds",
              len(incomplete) == 0,
              detail=f"{len(incomplete)} methods missing thresholds")

        # [5A.10] beta/se finite, pval in [0,1] for non-NA rows
        bad_numeric = []
        for col in ("beta", "se"):
            n_bad = (~np.isfinite(pd.to_numeric(all_rows[col], errors='coerce'))).sum()
            if n_bad > 0:
                bad_numeric.append((col, int(n_bad)))
        pvals = pd.to_numeric(all_rows["pval"], errors='coerce')
        n_bad_p = ((pvals < 0) | (pvals > 1)).sum()
        check("5A.10", "beta/se finite AND pval in [0,1]",
              len(bad_numeric) == 0 and n_bad_p == 0,
              detail=f"numeric={bad_numeric}, bad_p={n_bad_p}")

# [5A.11] MR-APSS Cor.SelectionBias=TRUE — search scripts
script_candidates = [AGENT_OUT / "scripts" / "step4_run_methods.R",
                     AGENT_OUT / "scripts" / "main_run_MR_methods.R"]
text = ""
for p in script_candidates:
    if p.exists():
        text += p.read_text() + "\n"
check("5A.11", "MR-APSS called with Cor.SelectionBias=TRUE",
      "Cor.SelectionBias = T" in text or "Cor.SelectionBias=TRUE" in text or
      "Cor.SelectionBias = TRUE" in text,
      detail="searched step4 / main_run_MR_methods script")

# [5A.12] CAUSE est_cause_params called once per pair before loop
# Heuristic: in the script, the call to est_cause_params should appear
# OUTSIDE a for-loop on Thresh
ok_cause = False
if text:
    # Find the for(Thresh ...) loop range and check est_cause_params is before it
    idx_estparams = text.find("est_cause_params")
    idx_threshloop = text.find("for(Thresh")
    if idx_estparams > 0 and (idx_threshloop < 0 or idx_estparams < idx_threshloop):
        ok_cause = True
check("5A.12", "est_cause_params called once per pair, outside threshold loop",
      ok_cause, detail="textual check on script")

# [5A.13] Sanity: no method's |beta| > 10x median for the same pair
sanity_violations = 0
if all_rows is not None and "beta" in all_rows.columns:
    grouped = all_rows.dropna(subset=["beta"]).groupby(["exposure", "outcome", "Threshold"])
    for key, group in grouped:
        if len(group) < 3: continue
        bvals = np.abs(pd.to_numeric(group["beta"], errors='coerce').dropna())
        if len(bvals) < 3: continue
        med = bvals.median()
        if med > 0 and (bvals > 10 * med).any():
            sanity_violations += 1
check("5A.13", "No method's |beta| is > 10x the median for the same pair",
      sanity_violations == 0,
      detail=f"{sanity_violations} pair-threshold groups violate")


## Task 6 — Aggregation and benchmark evaluation

In [ ]:
print()
print("=" * 70)
print("TASK 6 — Aggregation and benchmark evaluation")
print("=" * 70)

combined_path = AGENT_OUT / "results_combined.tsv"
combined = None
if combined_path.exists():
    combined = load_tsv_loose(combined_path)

# [6A.01] results_combined.tsv columns
required_cols = ["dataset", "exposure", "outcome", "method", "Threshold",
                 "nsnp", "beta", "se", "pval"]
ok_cols = combined is not None and all(c in combined.columns for c in required_cols)
check("6A.01", "results_combined.tsv has required columns",
      ok_cols,
      detail=f"columns: {list(combined.columns)[:10] if combined is not None else 'n/a'}")

# [6A.02] Failure rows have NA in beta/se/pval and a status column
if combined is not None and "status" in combined.columns:
    failed = combined[combined["status"].astype(str).str.upper() != "PASS"]
    ok = True
    if len(failed) > 0:
        ok = (failed[["beta", "se", "pval"]].isna().any(axis=1)).all()
    check("6A.02", "Failed rows have NA + status flag",
          ok, detail=f"{len(failed)} failed rows")
else:
    check("6A.02", "results_combined.tsv has a status column",
          False)

# [6A.03] evaluation_type1.tsv consistency
e_type1 = AGENT_OUT / "evaluation_type1.tsv"
if e_type1.exists():
    df1 = load_tsv_loose(e_type1)
    if df1 is not None and {"method","Threshold","n_pairs","n_significant","type1_rate"}.issubset(df1.columns):
        rate_check = (df1["n_significant"] / df1["n_pairs"] - df1["type1_rate"]).abs() < 1e-6
        check("6A.03", "evaluation_type1: type1_rate == n_significant/n_pairs",
              rate_check.all(), detail=f"{(~rate_check).sum()} bad rows")
    else:
        check("6A.03", "evaluation_type1.tsv has required columns",
              False)
else:
    check("6A.03", "evaluation_type1.tsv exists", False)

# [6A.04] n_pairs in evaluation_type1 matches actual count
if e_type1.exists() and combined is not None:
    df1 = load_tsv_loose(e_type1)
    if df1 is not None:
        bad = 0
        for _, row in df1.iterrows():
            actual = combined[
                (combined["method"] == row["method"]) &
                (combined["Threshold"].astype(float) == float(row["Threshold"])) &
                (pd.to_numeric(combined["pval"], errors='coerce').notna())
            ]
            if abs(len(actual) - int(row["n_pairs"])) > 2:
                bad += 1
        check("6A.04", "evaluation_type1 n_pairs matches actual finite-pval count",
              bad == 0, detail=f"{bad} mismatches (>2 row tolerance)")

# [6A.05] evaluation_accuracy.tsv exists with required columns (Dataset 6 only)
if DATASET == "dataset6":
    e_acc = AGENT_OUT / "evaluation_accuracy.tsv"
    if e_acc.exists():
        df2 = load_tsv_loose(e_acc)
        req_cols = {"method","Threshold","n_pairs","mean_bias","rmse","coverage_95"}
        check("6A.05", "evaluation_accuracy.tsv has required columns",
              df2 is not None and req_cols.issubset(df2.columns))
    else:
        check("6A.05", "evaluation_accuracy.tsv exists for Dataset 6",
              False)

# [6A.06] ranking_summary.tsv: 16 rows, has expected columns
r_sum = AGENT_OUT / "ranking_summary.tsv"
if r_sum.exists():
    dfr = load_tsv_loose(r_sum)
    ok = (dfr is not None and len(dfr) == 16 and
          "method" in dfr.columns and
          any("type1" in c.lower() for c in dfr.columns) and
          any("runtime" in c.lower() for c in dfr.columns))
    check("6A.06", "ranking_summary.tsv has 16 rows + Type I + runtime",
          ok, detail=f"rows={len(dfr) if dfr is not None else 0}")
else:
    check("6A.06", "ranking_summary.tsv exists", False)

# [6A.07] Recommended threshold used for ranking
if r_sum.exists():
    dfr = load_tsv_loose(r_sum)
    if dfr is not None and "Threshold" in dfr.columns and "method" in dfr.columns:
        ok = True
        for _, row in dfr.iterrows():
            expected = 5e-05 if "APSS" in str(row["method"]).upper() else 5e-08
            try:
                if abs(float(row["Threshold"]) - expected) > 1e-12:
                    ok = False
                    break
            except (TypeError, ValueError):
                ok = False
                break
        check("6A.07", "Ranking uses recommended threshold per method",
              ok, detail="MR-APSS@5e-05, others@5e-08")


## Summary

In [ ]:
print()
print("=" * 70)
print("SUMMARY")
print("=" * 70)

by_task = defaultdict(lambda: {"pass": 0, "fail": 0})
for r in results:
    by_task[r["task"]][r["status"].lower()] += 1

total_pass = 0
total_fail = 0
print(f"{'Task':<6} {'Pass':>6} {'Fail':>6} {'Total':>6} {'Rate':>8}")
print("-" * 38)
for t in sorted(by_task):
    p = by_task[t]["pass"]
    f = by_task[t]["fail"]
    total = p + f
    rate = (p / total * 100) if total else 0
    print(f"Task {t:<2} {p:>6} {f:>6} {total:>6} {rate:>7.1f}%")
    total_pass += p
    total_fail += f
total = total_pass + total_fail
overall_rate = (total_pass / total * 100) if total else 0
print("-" * 38)
print(f"{'TOTAL':<6} {total_pass:>6} {total_fail:>6} {total:>6} {overall_rate:>7.1f}%")

results_df = pd.DataFrame(results)
out_csv = AGENT_OUT / "verification_results.tsv"
try:
    AGENT_OUT.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(out_csv, sep='\t', index=False)
    print(f"\nDetailed results written to: {out_csv}")
except Exception as e:
    print(f"\nCould not write results: {e}")

# List failed checks for quick triage
print("\nFailed checks (first 30):")
fails = [r for r in results if r["status"] == "FAIL"]
for r in fails[:30]:
    detail = f" — {r['detail']}" if r['detail'] else ""
    print(f"  [{r['id']}] {r['check']}{detail}")
if len(fails) > 30:
    print(f"  ... and {len(fails) - 30} more")
